In [ ]:
import requests
import xarray as xr
import numpy as np
import scipy



In [ ]:
def download_and_open(url,typeOfKey = 'isobaricInhPa', filename="temp.grib2"):

  
  print("Downloading filtered data...")
  response = requests.get(url, stream=True)
  response.raise_for_status()

  with open(filename, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
      f.write(chunk)

      # 3. Open with xarray using the cfgrib engine
  ds = xr.open_dataset(filename, 
                       filter_by_keys={'typeOfLevel': typeOfKey },
                       engine='cfgrib')
  return ds  

In [42]:
url = 'https://noaa-nws-hafs-pds.s3.amazonaws.com/hfsa/20230927/12/17l.2023092712.hfsa.storm.atm.f024.grb2'

atm_args = dict(typeOfKey = 'isobaricInhPa',filename="Model_Data/temp_gribs/atm_temp.grib2" )
sfc_args = dict(typeOfKey = 'meanSea',filename="Model_Data/temp_gribs/sfc_temp.grb2" )


ds_sfc = download_and_open(url, **sfc_args)

Ignoring index file 'Model_Data/temp_gribs/sfc_temp.grb2.5b7b6.idx' older than GRIB file


In [ ]:
def get_sfc_center(ds):
    min_pressure = ds['prmsl'].min().values
    if min_pressure > 100500: # Pressure threshold (pa) 1005mb
        return None
    nan_mask = ds['prmsl'].isnull()
    ds_sfc = ds.where(~nan_mask, drop = True).isel(latitude = slice(50,-50), longitude = slice(50,-50))
    

    slp_threshold = np.quantile(ds_sfc['prmsl'], 0.01)
    slp_clusters = ds_sfc['prmsl'].where(ds_sfc['prmsl'] < slp_threshold)
    slp_clusters_mask = ~slp_clusters.isnull()

    clusters = scipy.ndimage.label(slp_clusters_mask)
    cluster_size = []
    for a in np.arange(clusters[1]):
        cluster_number = a + 1
        cluster_size.append(np.sum(np.where(clusters[0] == cluster_number, 1, 0)))

    
    largest_cluster = np.argmax(cluster_size) + 1
    center_y, center_x = scipy.ndimage.center_of_mass(clusters[0], largest_cluster)
    mslp = ds_sfc['prmsl'].where(clusters[0] == largest_cluster).min().values

    center_lat = ds_sfc.latitude.isel(latitude =int(np.round(center_y)))
    center_lon = ds_sfc.longitude.isel(longitude =int(np.round(center_x)))
    
    center_info = dict(center_lat = center_lat, center_lon = center_lon, mslp = mslp)
    return center_info


print(get_sfc_center(ds_sfc))

{'center_lat': <xarray.DataArray 'latitude' ()> Size: 8B
array(19.742596)
Coordinates:
    time        datetime64[ns] 8B 2023-09-27T12:00:00
    step        timedelta64[ns] 8B 1 days
    meanSea     float64 8B 0.0
    latitude    float64 8B 19.74
    valid_time  datetime64[ns] 8B 2023-09-28T12:00:00
Attributes:
    units:          degrees_north
    standard_name:  latitude
    long_name:      latitude, 'center_lon': <xarray.DataArray 'longitude' ()> Size: 8B
array(304.36600316)
Coordinates:
    time        datetime64[ns] 8B 2023-09-27T12:00:00
    step        timedelta64[ns] 8B 1 days
    meanSea     float64 8B 0.0
    longitude   float64 8B 304.4
    valid_time  datetime64[ns] 8B 2023-09-28T12:00:00
Attributes:
    units:          degrees_east
    standard_name:  longitude
    long_name:      longitude, 'mslp': array(100104.45, dtype=float32)}


In [ ]:
ds_sfc['prmsl'].plot()